In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_selection import SelectKBest, chi2
from sklearn.preprocessing import MinMaxScaler

In [4]:
from google.colab import files
uploaded = files.upload()

Saving cleaned_enron_spam_data.zip to cleaned_enron_spam_data.zip


In [5]:
import zipfile
import os

with zipfile.ZipFile('cleaned_enron_spam_data.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/')

os.listdir('/content/')

['.config',
 'cleaned_enron_spam_data.csv',
 'cleaned_enron_spam_data.zip',
 'sample_data']

In [7]:
df = pd.read_csv('/content/cleaned_enron_spam_data.csv')

print("Dataset Shape:", df.shape)
print("Columns:", df.columns.tolist())

df.head()

Dataset Shape: (33716, 7)
Columns: ['Message ID', 'Subject', 'Message', 'Spam/Ham', 'Date', 'Message_Length', 'Message_Length_Scaled']


,Message ID,Subject,Message,Spam/Ham,Date,Message_Length,Message_Length_Scaled
0,0,christmas tree farm pictures,NaN,0,1999-12-10,0,-0.343257
1,1,"vastar resources , inc .","gary , production from the high island larger ...",0,1999-12-13,4282,0.656405
2,2,calpine daily gas nomination,- calpine daily gas nomination 1 . doc,0,1999-12-14,38,-0.334385
3,3,re : issue,fyi - see note below - already done .\nstella\...,0,1999-12-14,1171,-0.069879
4,4,meter 7268 nov allocation,fyi .\n- - - - - - - - - - - - - - - - - - - -...,0,1999-12-14,1124,-0.080851


In [8]:
print(df['Spam/Ham'].value_counts())

Spam/Ham
1    17171
0    16545
Name: count, dtype: int64


In [9]:
df['Message_Length'] = df['Message'].fillna('').astype(str).apply(len)

df['Subject_Length'] = df['Subject'].fillna('').astype(str).apply(len)

df['Word_Count'] = df['Message'].fillna('').astype(str).apply(lambda x: len(x.split()))

df['Subject_Word_Count'] = df['Subject'].fillna('').astype(str).apply(lambda x: len(x.split()))

df['Has_Link'] = df['Message'].fillna('').astype(str).str.contains(r'http|www.', case=False).astype(int)

df['Has_Exclamation'] = df['Message'].fillna('').astype(str).str.contains('!').astype(int)

print(df[['Message_Length', 'Subject_Length', 'Word_Count',
          'Subject_Word_Count', 'Has_Link', 'Has_Exclamation']].head())

   Message_Length  Subject_Length  Word_Count  Subject_Word_Count  Has_Link  \
0               0              28           0                   4         0   
1            4282              24        1572                   5         0   
2              38              28           8                   4         0   
3            1171              10         344                   3         0   
4            1124              25         318                   4         0   

   Has_Exclamation  
0                0  
1                0  
2                0  
3                0  
4                0  


In [10]:
X = df[['Message_Length', 'Subject_Length',
        'Word_Count', 'Subject_Word_Count',
        'Has_Link', 'Has_Exclamation']]

y = df['Spam/Ham']

print("Feature Shape:", X.shape)
print("Target Shape:", y.shape)
print("\nTarget Values:")
print(y.value_counts())

Feature Shape: (33716, 6)
Target Shape: (33716,)

Target Values:
Spam/Ham
1    17171
0    16545
Name: count, dtype: int64


In [11]:
selector = SelectKBest(score_func=chi2, k=5)

X_new = selector.fit_transform(X, y)

selected_features = X.columns[selector.get_support()]

print("Selected Features:")
print(selected_features.tolist())

print("\nFeature Scores:")
for name, score in zip(X.columns, selector.scores_):
    print(name, ":", round(score, 2))

print("\nNew Feature Shape:", X_new.shape)

Selected Features:
['Message_Length', 'Subject_Length', 'Word_Count', 'Subject_Word_Count', 'Has_Exclamation']

Feature Scores:
Message_Length : 937314.93
Subject_Length : 10992.9
Word_Count : 290322.57
Subject_Word_Count : 3362.54
Has_Link : 2141.4
Has_Exclamation : 2840.66

New Feature Shape: (33716, 5)


In [12]:
selected_df = pd.DataFrame(X_new, columns=selected_features)

selected_df['Spam/Ham'] = y.values

selected_df.to_csv('task5_selected_features.csv', index=False)

print("Selected Features Dataset Saved!")
print("Dataset Shape:", selected_df.shape)
selected_df.head()

Selected Features Dataset Saved!
Dataset Shape: (33716, 6)


,Message_Length,Subject_Length,Word_Count,Subject_Word_Count,Has_Exclamation,Spam/Ham
0,0,28,0,4,0,0
1,4282,24,1572,5,0,0
2,38,28,8,4,0,0
3,1171,10,344,3,0,0
4,1124,25,318,4,0,0


In [13]:
from google.colab import files
files.download('task5_selected_features.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>